<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Error Analysis



In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

results_path = "/content/drive/MyDrive/efficient-finetuning-lora/results/lora_evaluation_100.csv"

error_data = pd.read_csv(results_path)

print("Shape:", error_data.shape)
print("\nColumns:")
print(error_data.columns.tolist())

error_data.head()

Shape: (37, 5)

Columns:
['instruction', 'input', 'reference', 'prediction', 'prediction_length']


,instruction,input,reference,prediction,prediction_length
0,"You are given a computer programming language,...",Python,Here is an example program written in Python t...,Here is an example Python program that demonst...,91
1,"Given the parameters of a triangle, find out i...",Side 1 = 4\nSide 2 = 6\nSide 3 = 8,The perimeter of a triangle is the sum of the ...,The perimeter of a triangle is the sum of all ...,95
2,Describe a landscape in three phrases.,NaN,1. Rolling hills dotted with wildflowers.\n2. ...,A beautiful mountain range with rolling hills ...,109
3,Summarize the difference between a tab and a s...,NaN,A tab and a space are both white space charact...,A tab is a small horizontal line that is used ...,116
4,Write an original joke,NaN,Why don't scientists trust atoms? Because they...,Why did the tomato turn red?\n\nBecause it saw...,74


In [3]:
for i, row in error_data.iterrows():

    print("\n" + "=" * 80)
    print(f"EXAMPLE {i}")
    print("=" * 80)

    print("\nINSTRUCTION:")
    print(row["instruction"])

    if pd.notna(row["input"]) and str(row["input"]).strip():
        print("\nINPUT:")
        print(row["input"])

    print("\nREFERENCE:")
    print(row["reference"])

    print("\nPREDICTION:")
    print(row["prediction"])


EXAMPLE 0

INSTRUCTION:
You are given a computer programming language, and you need to come up with an example program to demonstrate it.

INPUT:
Python

REFERENCE:
Here is an example program written in Python that will print the first 10 numbers in the Fibonacci sequence.

```python
# defining the function 
def fibonacci_seq(n):
    # assigning first two numbers of the sequence
    a = 0
    b = 1

    # check if the number of terms is zero
    if n <= 0:
        print("Invalid input. Enter a positive integer.")
    # check if the number of terms is one
    elif n == 1:
        print(a)
    else:
        #print the first two numbers
        print(a)
        print(b)
        # run a for loop to calculate the next numbers of the sequence
        for i in range(n-2):
            c = a + b
            a = b
            b = c
            print(c)

# Driver code to test the function
fibonacci_seq(10)
```

This program defines a function named `fibonacci_seq` which takes in one parameter `n

In [5]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=616f33180ab30be3d4ca9348d34fc20b7815806a8ab85668eb97e62cefb20820
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [7]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

error_results = []

for i, row in error_data.iterrows():

    reference = str(row["reference"])
    prediction = str(row["prediction"])

    scores = scorer.score(reference, prediction)

    error_results.append({
        "index": i,
        "instruction": row["instruction"],
        "reference": reference,
        "prediction": prediction,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
        "prediction_length": row["prediction_length"]
    })

error_df = pd.DataFrame(error_results)

error_df = error_df.sort_values(
    "rougeL",
    ascending=True
).reset_index(drop=True)

error_df.head(10)

,index,instruction,reference,prediction,rouge1,rouge2,rougeL,prediction_length
0,4,Write an original joke,Why don't scientists trust atoms? Because they...,Why did the tomato turn red?\n\nBecause it saw...,0.046512,0.000000,0.046512,74
1,32,Translate the statement below into French,Le rire est le meilleur médicament.,Laughter est la meilleure médecine. (Translati...,0.053571,0.018182,0.053571,101
2,23,Write a slogan for an animal rescue organization.,"""Rescuing animals, one life at a time!""","""Save the Animals - Save the Planet!"" This slo...",0.065041,0.000000,0.065041,116
3,7,Rewrite this sentence to make the words in bol...,We had a splendid weekend.,"""We had an outstanding weekend."" (Formal) \n\n...",0.075472,0.019231,0.075472,99
4,27,Create a grocery list for a week's worth of me...,Grocery List:\n- Spaghetti noodles\n- Marinara...,Here is a grocery list for a week's worth of m...,0.125984,0.016000,0.125984,82
5,22,Write a poem that must have 8 lines in it.,"A whisper on the wind, so soft yet clear, \nA ...","In the quiet of night,\nA whisper of wind and ...",0.215569,0.012121,0.131737,103
6,2,Describe a landscape in three phrases.,1. Rolling hills dotted with wildflowers.\n2. ...,A beautiful mountain range with rolling hills ...,0.136364,0.030769,0.136364,109
7,36,Create a story about a spacecraft crash-landin...,It was a clear day when the spacecraft made it...,"Once upon a time, in the vast expanse of space...",0.293023,0.084112,0.148837,112
8,6,Write a sentence to describe a sunset,A sunset paints the sky in a breathtaking pale...,"The sun slowly sinks below the horizon, castin...",0.269504,0.086331,0.156028,113
9,21,Summarize the given passage in no more than 25...,The automotive industry is using new technolog...,Automakers are innovating with new technologie...,0.203125,0.079365,0.156250,104


In [8]:
error_analysis = {
    4: {
        "category": "Incorrect",
        "observation": "Generated a different joke instead of following the given joke prompt/reference."
    },
    32: {
        "category": "Partial",
        "observation": "Translation contains mixed English and French instead of a fully consistent French translation."
    },
    23: {
        "category": "Valid variation",
        "observation": "Generated a relevant animal-rescue slogan, but used different wording from the reference."
    },
    7: {
        "category": "Valid variation",
        "observation": "Produced a reasonable formal rewrite with different wording from the reference."
    },
    27: {
        "category": "Needs review",
        "observation": "The generated grocery list appears relevant; low lexical overlap alone is not enough to call it an error."
    },
    22: {
        "category": "Needs review",
        "observation": "Creative poem generation should be checked for the required 8-line constraint rather than judged only by ROUGE."
    },
    2: {
        "category": "Valid variation",
        "observation": "Generated a relevant landscape description using different wording from the reference."
    },
    36: {
        "category": "Needs review",
        "observation": "Creative story generation can legitimately differ from the reference; semantic and instruction compliance should be checked."
    },
    6: {
        "category": "Valid variation",
        "observation": "Generated a relevant description of a sunset with different wording from the reference."
    },
    21: {
        "category": "Valid variation",
        "observation": "Generated a summary with different wording but appears related to the requested content."
    }
}

error_df["category"] = error_df["index"].map(
    lambda x: error_analysis.get(
        x, {"category": "Not reviewed"}
    )["category"]
)

error_df["observation"] = error_df["index"].map(
    lambda x: error_analysis.get(
        x, {"observation": "Not reviewed"}
    )["observation"]
)

reviewed_errors = error_df[
    error_df["index"].isin(error_analysis.keys())
][[
    "index",
    "rouge1",
    "rouge2",
    "rougeL",
    "category",
    "observation"
]]

reviewed_errors

,index,rouge1,rouge2,rougeL,category,observation
0,4,0.046512,0.000000,0.046512,Incorrect,Generated a different joke instead of followin...
1,32,0.053571,0.018182,0.053571,Partial,Translation contains mixed English and French ...
2,23,0.065041,0.000000,0.065041,Valid variation,"Generated a relevant animal-rescue slogan, but..."
3,7,0.075472,0.019231,0.075472,Valid variation,Produced a reasonable formal rewrite with diff...
4,27,0.125984,0.016000,0.125984,Needs review,The generated grocery list appears relevant; l...
5,22,0.215569,0.012121,0.131737,Needs review,Creative poem generation should be checked for...
6,2,0.136364,0.030769,0.136364,Valid variation,Generated a relevant landscape description usi...
7,36,0.293023,0.084112,0.148837,Needs review,Creative story generation can legitimately dif...
8,6,0.269504,0.086331,0.156028,Valid variation,Generated a relevant description of a sunset w...
9,21,0.203125,0.079365,0.156250,Valid variation,Generated a summary with different wording but...


In [11]:
category_counts = (
    reviewed_errors["category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["category", "count"]

category_counts

,category,count
0,Valid variation,5
1,Needs review,3
2,Partial,1
3,Incorrect,1


In [12]:
review_indices = [27, 22, 36]

for idx in review_indices:

    row = error_data[error_data["instruction"].index == idx].iloc[0]

    print("\n" + "=" * 80)
    print(f"EXAMPLE {idx}")
    print("=" * 80)

    print("\nINSTRUCTION:")
    print(row["instruction"])

    if pd.notna(row["input"]) and str(row["input"]).strip():
        print("\nINPUT:")
        print(row["input"])

    print("\nREFERENCE:")
    print(row["reference"])

    print("\nPREDICTION:")
    print(row["prediction"])


EXAMPLE 27

INSTRUCTION:
Create a grocery list for a week's worth of meals.

INPUT:
Meals for the week: Spaghetti, tacos, grilled chicken, stir-fry

REFERENCE:
Grocery List:
- Spaghetti noodles
- Marinara sauce
- Ground beef
- Taco shells
- Taco seasoning
- Shredded cheese
- Lettuce
- Tomatoes
- Salsa
- Chicken breasts
- Salt
- Pepper
- Mixed vegetables (broccoli, carrots, snap peas)
- Soy sauce
- Brown sugar
- Rice

Optional ingredients to enhance the meals:
- Garlic
- Onions
- Olive oil
- Hot sauce
- Sour cream.

PREDICTION:
Here is a grocery list for a week's worth of meals:

- Spaghetti (1 lb)
- Tacos (2-3 servings)
- Grilled Chicken (4-5 pieces)
- Stir-Fry (1-2 servings)

Note: This list can be adjusted based on your personal preferences and dietary needs. For example, you could add more vegetables or use different types of meat to make it more diverse. Also, consider buying in bulk if possible as this will save money. Enjoy your meal! 🍽️😊✨ #GroceryList #HealthyEating #MealPrep #

In [15]:
error_analysis.update({
    27: {
        "category": "Incomplete",
        "observation": "The model listed meals instead of providing the requested grocery ingredients."
    },
    22: {
        "category": "Instruction-following error",
        "observation": "The model generated a poem but did not follow the explicit 8-line constraint."
    },
    36: {
        "category": "Valid variation",
        "observation": "The model generated a relevant spacecraft story; different wording and storyline are acceptable for a creative-writing task."
    }
})

reviewed_errors = error_df[
    error_df["index"].isin(error_analysis.keys())
].copy()

reviewed_errors["category"] = reviewed_errors["index"].map(
    lambda x: error_analysis[x]["category"]
)

reviewed_errors["observation"] = reviewed_errors["index"].map(
    lambda x: error_analysis[x]["observation"]
)

category_counts = (
    reviewed_errors["category"]
    .value_counts()
    .reset_index()
)

category_counts.columns = ["category", "count"]

print("Final Error Analysis:")
display(category_counts)

print("\nDetailed Analysis:")
display(
    reviewed_errors[
        [
            "index",
            "rouge1",
            "rouge2",
            "rougeL",
            "category",
            "observation"
        ]
    ]
)

Final Error Analysis:


,category,count
0,Valid variation,6
1,Incorrect,1
2,Partial,1
3,Incomplete,1
4,Instruction-following error,1



Detailed Analysis:


,index,rouge1,rouge2,rougeL,category,observation
0,4,0.046512,0.000000,0.046512,Incorrect,Generated a different joke instead of followin...
1,32,0.053571,0.018182,0.053571,Partial,Translation contains mixed English and French ...
2,23,0.065041,0.000000,0.065041,Valid variation,"Generated a relevant animal-rescue slogan, but..."
3,7,0.075472,0.019231,0.075472,Valid variation,Produced a reasonable formal rewrite with diff...
4,27,0.125984,0.016000,0.125984,Incomplete,The model listed meals instead of providing th...
5,22,0.215569,0.012121,0.131737,Instruction-following error,The model generated a poem but did not follow ...
6,2,0.136364,0.030769,0.136364,Valid variation,Generated a relevant landscape description usi...
7,36,0.293023,0.084112,0.148837,Valid variation,The model generated a relevant spacecraft stor...
8,6,0.269504,0.086331,0.156028,Valid variation,Generated a relevant description of a sunset w...
9,21,0.203125,0.079365,0.156250,Valid variation,Generated a summary with different wording but...
